Start by importing the required packages.

In [2]:
import re
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

Below are dictionaries for name conversions that happen later on in the code.

In [4]:
IOC_TO_COUNTRY = {
    "AFG": "Afghanistan",           "ALB": "Albania",
    "ALG": "Algeria",               "AND": "Andorra",
    "ANG": "Angola",                "ANT": "Antigua and Barbuda",
    "ARG": "Argentina",             "ARM": "Armenia",
    "ARU": "Aruba",                 "ASA": "American Samoa",
    "AUS": "Australia",             "AUT": "Austria",
    "AZE": "Azerbaijan",            "BAH": "Bahamas",
    "BAN": "Bangladesh",            "BAR": "Barbados",
    "BDI": "Burundi",               "BEL": "Belgium",
    "BEN": "Benin",                 "BER": "Bermuda",
    "BHU": "Bhutan",                "BIH": "Bosnia and Herzegovina",
    "BIZ": "Belize",                "BLR": "Belarus",
    "BOL": "Bolivia",               "BOT": "Botswana",
    "BRA": "Brazil",                "BRN": "Bahrain",
    "BRU": "Brunei",                "BUL": "Bulgaria",
    "BUR": "Burkina Faso",          "CAF": "Central African Republic",
    "CAM": "Cambodia",              "CAN": "Canada",
    "CAY": "Cayman Islands",        "CGO": "Republic of the Congo",
    "CHA": "Chad",                  "CHI": "Chile",
    "CHN": "China",                 "CIV": "Côte d'Ivoire",
    "CMR": "Cameroon",              "COD": "Democratic Republic of the Congo",
    "COK": "Cook Islands",          "COL": "Colombia",
    "COM": "Comoros",               "CPV": "Cape Verde",
    "CRC": "Costa Rica",            "CRO": "Croatia",
    "CUB": "Cuba",                  "CYP": "Cyprus",
    "CZE": "Czech Republic",        "DEN": "Denmark",
    "DJI": "Djibouti",              "DMA": "Dominica",
    "DOM": "Dominican Republic",    "ECU": "Ecuador",
    "EGY": "Egypt",                 "ESA": "El Salvador",
    "ESP": "Spain",                 "EST": "Estonia",
    "ETH": "Ethiopia",              "FIJ": "Fiji",
    "FIN": "Finland",               "FRA": "France",
    "FSM": "Micronesia",            "GAB": "Gabon",
    "GAM": "Gambia",                "GBR": "United Kingdom",
    "GBS": "Guinea-Bissau",         "GEO": "Georgia",
    "GER": "Germany",               "GHA": "Ghana",
    "GRE": "Greece",                "GRN": "Grenada",
    "GUA": "Guatemala",             "GUI": "Guinea",
    "GUM": "Guam",                  "GUY": "Guyana",
    "HAI": "Haiti",                 "HKG": "Hong Kong",
    "HON": "Honduras",              "HUN": "Hungary",
    "INA": "Indonesia",             "IND": "India",
    "IRI": "Iran",                  "IRL": "Ireland",
    "ISL": "Iceland",               "ISR": "Israel",
    "ISV": "Virgin Islands",        "ITA": "Italy",
    "JAM": "Jamaica",               "JOR": "Jordan",
    "JPN": "Japan",                 "KAZ": "Kazakhstan",
    "KEN": "Kenya",                 "KGZ": "Kyrgyzstan",
    "KIR": "Kiribati",              "KOR": "South Korea",
    "KOS": "Kosovo",                "KSA": "Saudi Arabia",
    "KUW": "Kuwait",                "LAO": "Laos",
    "LAT": "Latvia",                "LBA": "Libya",
    "LBN": "Lebanon",               "LBR": "Liberia",
    "LCA": "Saint Lucia",           "LES": "Lesotho",
    "LIE": "Liechtenstein",         "LTU": "Lithuania",
    "LUX": "Luxembourg",            "MAD": "Madagascar",
    "MAR": "Morocco",               "MAS": "Malaysia",
    "MAW": "Malawi",                "MDA": "Moldova",
    "MDV": "Maldives",              "MEX": "Mexico",
    "MGL": "Mongolia",              "MHL": "Marshall Islands",
    "MKD": "North Macedonia",       "MLI": "Mali",
    "MLT": "Malta",                 "MON": "Monaco",
    "MOZ": "Mozambique",            "MRI": "Mauritius",
    "MTN": "Mauritania",            "MYA": "Myanmar",
    "NAM": "Namibia",               "NCA": "Nicaragua",
    "NED": "Netherlands",           "NEP": "Nepal",
    "NGR": "Nigeria",               "NIG": "Niger",
    "NOR": "Norway",                "NRU": "Nauru",
    "NZL": "New Zealand",           "OMA": "Oman",
    "PAK": "Pakistan",              "PAN": "Panama",
    "PAR": "Paraguay",              "PER": "Peru",
    "PHI": "Philippines",           "PLE": "Palestine",
    "PLW": "Palau",                 "PNG": "Papua New Guinea",
    "POL": "Poland",                "POR": "Portugal",
    "PRK": "North Korea",           "PUR": "Puerto Rico",
    "QAT": "Qatar",                 "ROU": "Romania",
    "RSA": "South Africa",          "RUS": "Russia",
    "RWA": "Rwanda",                "SAM": "Samoa",
    "SEN": "Senegal",               "SEY": "Seychelles",
    "SGP": "Singapore",             "SKN": "Saint Kitts and Nevis",
    "SLE": "Sierra Leone",          "SLO": "Slovenia",
    "SMR": "San Marino",            "SOL": "Solomon Islands",
    "SOM": "Somalia",               "SRB": "Serbia",
    "SRI": "Sri Lanka",             "SSD": "South Sudan",
    "STP": "São Tomé and Príncipe", "SUD": "Sudan",
    "SUI": "Switzerland",           "SUR": "Suriname",
    "SVK": "Slovakia",              "SWE": "Sweden",
    "SWZ": "Eswatini",              "SYR": "Syria",
    "TAN": "Tanzania",              "TGA": "Tonga",
    "THA": "Thailand",              "TJK": "Tajikistan",
    "TKM": "Turkmenistan",          "TLS": "Timor-Leste",
    "TOG": "Togo",                  "TPE": "Taiwan",
    "TTO": "Trinidad and Tobago",   "TUN": "Tunisia",
    "TUR": "Turkey",                "TUV": "Tuvalu",
    "UAE": "United Arab Emirates",  "UGA": "Uganda",
    "UKR": "Ukraine",               "URU": "Uruguay",
    "USA": "United States",         "UZB": "Uzbekistan",
    "VAN": "Vanuatu",               "VEN": "Venezuela",
    "VIE": "Vietnam",               "VIN": "Saint Vincent and the Grenadines",
    "YEM": "Yemen",                 "ZAM": "Zambia",
    "ZIM": "Zimbabwe",       
}

LOCATION_CORRECTIONS = {
    "EschAlzette, Luxembourg":  "Esch-sur-Alzette, Luxembourg",
    "Mungia-Laukariz, Spain":   "Mungia, Spain",
    "Qian Daohu, China":        "Qiandaohu, China",
    "RisskovAarhus, Denmark":   "Aarhus, Denmark",
    "Sharm ElSheikh, Egypt":    "Sharm el-Sheikh, Egypt",
}

CHALL_NAME_FIXES = {
    "Cordoba":        "Cordoba, Argentina",
    "Santiago":       "Santiago, Chile",
    "Los Cabos":      "Los Cabos, Baja California Sur, Mexico",
    "Cali":           "Cali, Colombia",
    "Lima":           "Lima, Peru",
    "Bogota":         "Bogota, Colombia",
    "Istanbul TTF":   "Istanbul",
    "Tigre":          "Tigre, Argentina",
    "Košice":         "Košice, Slovakia",
    "Villa Maria":    "Villa Maria, Argentina",
    "Montemar":       "Montemar, Alicante, Spain",
    "Mérida":         "Mérida, Mexico",
    "Brasilia":       "Brasília (Federal District), Brazil",
    "Tunis":          "Tunis City, Tunisia",
    "Salinas":        "Salinas, Ecuador",
    "Fairfield":      "Fairfield CA, United States",
    "Sumter":         "Sumter SC, United States",
    "Leon":           "Leon, Mexico",
}

TOURNEY_NAME_MAP = {
    "Australian Open":  "Melbourne",
    "Roland Garros":    "Paris",
    "Queen's Club":     "London",
    "Us Open":          "New York",
    "Tour Finals":      "Turin",
    "NextGen Finals":   "Jeddah",
    "Next Gen Finals":  "Jeddah",
    "Cordoba":          "Cordoba, Argentina",
    "Los Cabos":        "Los Cabos, Baja California Sur, Mexico",
    "Santiago":         "Santiago, Chile",
    "Paris Olympics":   "Paris",
}

LEVEL_LABELS = {
    "G": "Grand Slam",
    "M": "Masters",
    "F": "Finals",
    "A": "Tour Event",
    "C": "Challenger",
}

LEVEL_WEIGHTS = {
    "Grand Slam": 6,
    "Finals": 5,
    "Masters": 4,
    "Tour Event": 3,
    "Challenger": 2,
    "Futures": 1,
}

In [5]:
def parse_futures_name(name: str) -> str:
    name = re.sub(r'^M\d+(\+H)?\s+', '', name)  # remove M15, M25, M15+H, M25+H prefix
    name = re.sub(r'\s*\+H$', '', name)           # remove trailing +H (2026 format)
    name = re.sub(r'\s*\(.*?\)', '', name)         # remove anything in parentheses e.g. (moved from Coquimbo)
    return name.strip()

In [6]:
def strip_trailing_number(name: str) -> str:
    return re.sub(r'\s+\d+$', '', name).strip()

In [7]:
def clean_atp(year):
    df = pd.read_csv(f"atp_matches_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
    
    df = df[~df['tourney_name'].str.contains('Davis Cup|Laver Cup', case=False, na=False)]
    df["tourney_name"] = df["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
    df["tourney_name"] = df["tourney_name"].apply(strip_trailing_number)
    
    name_map = {**TOURNEY_NAME_MAP, "Canada": "Montreal" if year % 2 == 0 else "Toronto"}
    df["tourney_name"] = df["tourney_name"].replace(name_map)
    
    df["tourney_name"] = df["tourney_name"].apply(
        lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" and year == 2023
        else ["Sydney", "Perth"] if x == "United Cup"
        else [x]
    )
    df = df.explode("tourney_name")
    
    df["year"] = year
    df = df.drop_duplicates(subset=["tourney_id", "tourney_name", "year"])
    return df

In [8]:
def clean_atp_chall(year):
    df = pd.read_csv(f"atp_matches_qual_chall_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]

    # Keep challenger events only
    df = df[df["tourney_level"] == "C"].copy()

    # Remove "CH" as a standalone token, then fix ambiguous city names
    df["tourney_name"] = (
        df["tourney_name"]
        .str.replace(r"\bCH\b", "", regex=True)
        .str.strip()
        .apply(strip_trailing_number)
        .replace(CHALL_NAME_FIXES)
    )

    df["year"] = year
    df = df.drop_duplicates(subset=["tourney_id", "tourney_name", "year"])
    return df

In [9]:
def clean_atp_futures(year):
    df = pd.read_csv(f"atp_matches_futures_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]

    df["tourney_name"] = df["tourney_name"].apply(parse_futures_name)

    df["country_code"] = df["tourney_id"].str.split("-").str[3]
    df["tourney_name"] = df["tourney_name"] + ", " + df["country_code"].map(IOC_TO_COUNTRY).fillna(df["country_code"])
    df = df.drop(columns=["country_code"])

    df["year"] = year
    df = df.drop_duplicates(subset=["tourney_id", "tourney_name", "year"])
    return df

In [10]:
years = [2023, 2024, 2025, 2026]

atp_df     = pd.concat([clean_atp(y)         for y in years], ignore_index=True)
futures_df = pd.concat([clean_atp_futures(y) for y in years], ignore_index=True)
chall_df   = pd.concat([clean_atp_chall(y)   for y in years], ignore_index=True)

all_tournaments = pd.concat([atp_df, futures_df, chall_df], ignore_index=True)
all_tournaments = all_tournaments.drop_duplicates(subset=["tourney_id", "tourney_name", "year"])
all_tournaments["tourney_name"] = all_tournaments["tourney_name"].replace(LOCATION_CORRECTIONS)

In [11]:
GEOCODE_CACHE_FILE = "geocode_cache.csv"

try:
    cache_df = pd.read_csv(GEOCODE_CACHE_FILE)
except FileNotFoundError:
    cache_df = pd.DataFrame({
        "tourney_name": pd.Series(dtype="str"),
        "latitude": pd.Series(dtype="float64"),
        "longitude": pd.Series(dtype="float64"),
    })

known_cities = set(cache_df["tourney_name"])
unique_cities = all_tournaments["tourney_name"].unique()
new_cities = [c for c in unique_cities if c not in known_cities]

print(f"{len(unique_cities)} unique locations total, {len(new_cities)} new — geocoding those only.")

geolocator = Nominatim(user_agent="Professional_Men's_Tennis2023-2026_Analysis")
geocode    = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

location_data = {}
for city in new_cities:
    loc = geocode(city)
    if loc:
        location_data[city] = (loc.latitude, loc.longitude)
    else:
        print(f"Could not geocode: {city}")

new_locations_df = pd.DataFrame.from_dict(
    location_data, orient="index", columns=["latitude", "longitude"]
)
new_locations_df.index.name = "tourney_name"
new_locations_df = new_locations_df.reset_index()

new_df = (
    pd.concat([cache_df, new_locations_df], ignore_index=True)
    .drop_duplicates(subset="tourney_name", keep="last")
)
new_df.to_csv(GEOCODE_CACHE_FILE, index=False)

876 unique locations total, 0 new — geocoding those only.


C:\Users\Utlisateur\AppData\Local\Temp\ipykernel_32220\2836024210.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([cache_df, new_locations_df], ignore_index=True)


In [12]:
def add_level_columns(df):
    df = df.copy()
    # Anything not in LEVEL_LABELS is a futures event (numeric 15/25 level)
    df["tourney_level"] = df["tourney_level"].apply(lambda x: LEVEL_LABELS.get(x, "Futures"))
    df["Level Weight"] = df["tourney_level"].map(LEVEL_WEIGHTS).fillna(1).astype(int)
    return df

def add_date_columns(df):
    df = df.copy()
    df["tourney_date"] = pd.to_datetime(df["tourney_date"].astype(str), format="%Y%m%d")
    df["Month Name"] = df["tourney_date"].dt.strftime("%B")
    df["Month Number"] = df["tourney_date"].dt.month
    return df

In [13]:
def add_draw_columns(df):
    df = df.copy()
    df["Draw Weight"] = df["draw_size"]
    df["Draw Size"] = df["draw_size"].astype(str) + " Draw"
    return df

In [14]:
final_df = all_tournaments.merge(
    new_df[["tourney_name", "latitude", "longitude"]],
    on="tourney_name",
    how="left",
)
final_df = (
    final_df
    .pipe(add_level_columns)
    .pipe(add_date_columns)
    .pipe(add_draw_columns)
    .drop_duplicates(subset=["tourney_id", "tourney_name"])
    .rename(columns={
        "tourney_id": "Tournament ID",
        "tourney_name": "Tournament City",
        "surface": "Surface",
        "tourney_level": "Tournament Level",
        "tourney_date": "Tournament Date",
        "latitude": "Latitude",
        "longitude": "Longitude",
    })
)
final_df
final_df.to_csv("atp_all_locations.csv", index=False)

In [15]:
print(all_tournaments["tourney_name"])

0        Brisbane
1          Sydney
2           Perth
3        Adelaide
4            Pune
          ...    
2939    Centurion
2940    Heilbronn
2941      Perugia
2942    Prostejov
2943        Tyler
Name: tourney_name, Length: 2944, dtype: object
